# Phase 1: Document Vectorization (LangChain Edition)

This notebook demonstrates the first stage of the RAG pipeline using LangChain abstractions: **Ingestion and Storage**.

We will cover:
1. **Extraction**: Reading text from a PDF into LangChain `Document` objects.
2. **Chunking**: Splitting `Document`s into manageable segments.
3. **Embedding**: Converting text into semantic vectors using `HuggingFaceEmbeddings`.
4. **Indexing**: Saving segments to a local ChromaDB store via LangChain.

In [ ]:
import os
import sys

# Add the src directory to the python path so we can import our modules
sys.path.append(os.path.abspath('../src'))

from ingest.parser import extract_text_from_pdf
from ingest.chunker import chunk_documents
from rag.retriever import VectorRetriever

print("Modules loaded successfully.")

## 1. Text Extraction

We'll use LangChain's `PDFPlumberLoader` to extract raw text from a sample PDF.

> **Note**: This natively returns a list of LangChain `Document` objects, including metadata like the source page number!

In [ ]:
pdf_path = "../data/raw/sample.pdf"

if not os.path.exists(pdf_path):
    print(f"Error: {pdf_path} not found. Please ensure a sample PDF exists in data/raw/")
else:
    documents = extract_text_from_pdf(pdf_path)
    print(f"Extracted {len(documents)} pages from the PDF.")
    print("\n--- Sample Text (First 1000 chars of Page 1) ---")
    print(documents[0].page_content[:1000])

## 2. Text Chunking

Raw text is too long for LLMs to process efficiently. We split our LangChain `Document`s into smaller overlapping chunks.

In [ ]:
chunk_size = 500
chunk_overlap = 100
chunks = chunk_documents(documents, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
print(f"Split documents into {len(chunks)} chunks.")

print("\n--- Chunk 1 ---")
print(chunks[0].page_content)

print("\n--- Chunk 2 ---")
print(chunks[1].page_content)

## 3. Vector Storage (LangChain Chroma)

Now we convert these chunks into embeddings using a local `HuggingFaceEmbeddings` model and save them to our local Chroma vector store via LangChain.

In [ ]:
retriever = VectorRetriever()

# Add chunks to store
retriever.add_documents(chunks)

print(f"Successfully indexed {retriever.get_collection_count()} items in the vector store.")

## 4. Verification

Let's check if we can retrieve something back.

In [ ]:
results = retriever.vectorstore.get(limit=5)
print(f"Retrieved {len(results['ids'])} items from the store.")
